In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
MERGE INTO silver.characters_summary AS target
USING (
  SELECT
    raw_characters_id AS snapshot_id,
    sha1(concat_ws('|', lower(trim(character.character.name)), lower(trim(character.character.world)), source_file_date)) AS snapshot_character_id,
    character.character.name AS name,
    information.api.version AS api_version,
    information.api.release AS api_release,
    information.api.commit AS api_commit,
    to_timestamp(information.timestamp) AS api_processed_at,
    information.tibia_urls AS tibia_urls,
    information.status.error AS error,
    information.status.http_code AS http_code,
    information.status.message AS message,
    source_file_date,
    source_file_path,
    source_file_modified_at,
    ingested_at,
    current_timestamp() AS processed_at
  FROM tibia_analytics.bronze.raw_characters
) AS source
ON target.snapshot_id = source.snapshot_id
WHEN NOT MATCHED THEN
  INSERT *;